## PAM Clustering Analysis for Multilingual Text Corpora

This notebook reproduces the Partitioning Around Medoids (PAM) clustering analysis for multilingual text corpora using Delta-style representations and distance measures.

**Reference:** [ExplainingDelta](https://github.com/schtepf/ExplainingDelta) (Evert et al., 2017).

### Overview

- **Input representation:** `data/TfMatrix{LANG}.json` stores the term–document matrix of relative frequencies $f_i$, with metadata columns `Count` and `Word`, followed by one column per document. For English, German and French, precomputed term–document matrices from the ExplainingDelta repository (Evert et al.) are used; term–document matrices for Russian are precomputed and described in the notebook `russian_tf.ipynb`.

- **Transformations:**  
  - centred z-scores $ (f_i - \mu) / \sigma $  
  - uncentred standard scores $ f_i / \sigma $
- **Distances:** L1, L2, cosine (in z-space); Jensen–Shannon (on probability and uncentered distributions); rank-turbulence divergence (on rank distributions).
- **Evaluation:** K-medoids (PAM) with a fixed random seed; quality measured by the adjusted Rand index over a log-spaced MFW schedule.

In [4]:
# Imports
import os
import numpy as np
import pandas as pd
import tqdm

from scipy.stats import norm, rankdata
from scipy.special import rel_entr  # Correct zero handling in KL divergence
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import (
    adjusted_rand_score,
    accuracy_score,
    balanced_accuracy_score,
)
from collections import Counter

# Random seed for reproducibility
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

## Jensen–Shannon Divergence with Zero Handling

The Jensen–Shannon divergence is defined for probability distributions (unit mass) and is undefined when entries are zero, since \(\log 0\) is not defined.

**Procedure:** Laplace smoothing (pseudocounts) is applied: a small constant \(\varepsilon\) is added to all components before normalisation, so that no probability is zero whilst the relative structure of the distribution is preserved.

**Zero-handling convention:**
- \(\varepsilon\) (default \(10^{-10}\)) is added to all components before normalisation (Laplace smoothing).
- Numerical stability and reproducibility are thereby ensured.
- The implementation uses SciPy’s `rel_entr` for the KL terms; the limiting value \(0\log(0/y)=0\) is adopted when \(p=0\) or \(q=0\).

In [5]:
def jensen_shannon_safe(p, q, epsilon=1e-10):
    """Jensen–Shannon distance with deterministic handling of zeros.

    We treat inputs as non-negative weight vectors over a fixed vocabulary. The procedure is:
    (i) normalise each vector to unit mass, (ii) add a small pseudocount `epsilon` to avoid
    undefined log terms, (iii) renormalise, and (iv) compute the square-rooted JS divergence.

    This is compatible with the probabilistic reading of non-negative (e.g., uncentred)
    standard-score vectors discussed in the accompanying paper, and relies on SciPy's
    `rel_entr` convention that the limit value of \(0\log 0\) is 0.
    """
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    
    # Normalise to unit mass
    p_sum = p.sum()
    q_sum = q.sum()
    
    if p_sum > 0:
        p = p / p_sum
    else:
        p = np.ones_like(p) / len(p)
    
    if q_sum > 0:
        q = q / q_sum
    else:
        q = np.ones_like(q) / len(q)
    
    # Laplace smoothing: add epsilon before normalisation (zero-handling convention)
    p = p + epsilon
    q = q + epsilon
    
    # Renormalise
    p = p / p.sum()
    q = q / q.sum()
    
    # Jensen–Shannon divergence via rel_entr (limit 0*log(0)=0)
    m = (p + q) / 2.0
    left = rel_entr(p, m)   # KL(P || M)
    right = rel_entr(q, m)  # KL(Q || M)
    js_div = 0.5 * np.sum(left) + 0.5 * np.sum(right)
    
    return np.sqrt(js_div)  # JS distance (square root of JS divergence)

### Rank-Turbulence Divergence

The following section implements **rank-turbulence divergence** (RTD): a divergence between two rank distributions induced by frequency vectors (ranks assigned in descending order of frequency), with parameter α and normalization by the fully disjoint case. Types absent in one distribution receive a final tie-rank. The resulting metric is used as a document-to-document distance over word ranks and is supplied to PAM and k-NN.

**Reference:** [Dodds et al. (2023), *Rank-turbulence divergence as a general purpose diversity metric*](https://link.springer.com/content/pdf/10.1140/epjds/s13688-023-00400-x.pdf).

In [6]:
def rank_turbulence(f1, f2, alpha=1.0, axis=0):
    """Rank–turbulence divergence (RTD), following Dodds et al. (2023).

    Summary of the construction:
    - Ranks are assigned in descending order of frequency/weight among present types (f > 0).
    - Types absent in one vector but present in the other are given a "final tie-rank":
        r_missing_in_1 = N1 + N2/2
        r_missing_in_2 = N2 + N1/2
      where N1 = |{f1 > 0}|, N2 = |{f2 > 0}|.
    - Divergence is:
        D = (1/Norm) * (α+1)/α * Σ_{τ in union} | (1/r1)^α - (1/r2)^α |^{1/(α+1)}
    - The normalization Norm is the value of the "raw" sum for a fully disjoint case:
        Norm = (α+1)/α * [ Σ_{τ in Ω1} | (1/r_{τ,1})^α - (1/r_missing_in_1)^α |^{1/(α+1)}
                             + Σ_{τ in Ω2} | (1/r_missing_in_2)^α - (1/r_{τ,2})^α |^{1/(α+1)} ]

    Parameters:
    -----------
    f1, f2 : array-like
        Input frequency vectors or arrays
    alpha : float, default=1.0
        Alpha parameter controlling sensitivity to rank differences.
        Smaller alpha: more sensitive to differences in top-ranked items
        Larger alpha: more sensitive to differences across all ranks
    axis : int, default=0
        Axis along which to compute ranks (for vectorized operations)

    Returns:
    --------
    float or array
        Rank-turbulence divergence value(s)
    """
    if alpha == 0:
        raise ValueError("alpha=0 requires limit-case handling (log-form), which is not implemented here.")

    f1 = np.asarray(f1)
    f2 = np.asarray(f2)
    if f1.shape != f2.shape:
        raise ValueError("f1 and f2 must have the same shape (vectorized over the same types/words).")

    # Move the axis to compare over (usually the vocabulary axis) to the last dimension for batching
    f1m = np.moveaxis(f1, axis, -1)
    f2m = np.moveaxis(f2, axis, -1)

    # Reshape to (batch_size..., vocab_size)
    batch_shape = f1m.shape[:-1]
    V = f1m.shape[-1]
    f1b = f1m.reshape(-1, V)
    f2b = f2m.reshape(-1, V)

    out = np.empty((f1b.shape[0],), dtype=float)
    c = (alpha + 1.0) / alpha
    p = 1.0 / (alpha + 1.0)

    for i in range(f1b.shape[0]):
        a = f1b[i]
        b = f2b[i]

        present1 = a > 0   # Types present in f1
        present2 = b > 0   # Types present in f2
        union = present1 | present2  # All types present in either

        N1 = int(present1.sum())
        N2 = int(present2.sum())

        # If neither vector has any types, divergence is undefined; return 0
        if (N1 == 0) and (N2 == 0):
            out[i] = 0.0
            continue

        # "Final tie-rank" values for absent types (Dodds et al., 2023)
        r_missing_in_1 = N1 + N2 / 2.0
        r_missing_in_2 = N2 + N1 / 2.0

        # Ranks among present types; lower rank means higher frequency
        r1 = np.empty(V, dtype=float)
        r2 = np.empty(V, dtype=float)

        # Rank in decreasing order; ties get the average rank as standard
        if N1 > 0:
            r1[present1] = rankdata(-a[present1], method="average")
        # For types present in f2 but not in f1
        r1[~present1 & present2] = r_missing_in_1

        if N2 > 0:
            r2[present2] = rankdata(-b[present2], method="average")
        # For types present in f1 but not in f2
        r2[~present2 & present1] = r_missing_in_2

        # For types present in neither (both zeros), assign NaN since they are not considered
        r1[~union] = np.nan
        r2[~union] = np.nan

        inv_r1 = 1.0 / r1[union]
        inv_r2 = 1.0 / r2[union]

        # Raw RTD sum over the union of present types
        raw = c * np.sum(np.abs(inv_r1**alpha - inv_r2**alpha) ** p)

        # Disjoint-case normalization:
        # For types only in f1, compare to r_missing_in_1; for types only in f2, to r_missing_in_2
        inv_m1 = 1.0 / r_missing_in_1 if r_missing_in_1 != 0 else 0.0
        inv_m2 = 1.0 / r_missing_in_2 if r_missing_in_2 != 0 else 0.0

        norm_part1 = 0.0
        if N1 > 0:
            inv1 = 1.0 / r1[present1]
            norm_part1 = np.sum(np.abs(inv1**alpha - inv_m1**alpha) ** p)

        norm_part2 = 0.0
        if N2 > 0:
            inv2 = 1.0 / r2[present2]
            norm_part2 = np.sum(np.abs(inv_m2**alpha - inv2**alpha) ** p)

        norm = c * (norm_part1 + norm_part2)
        out[i] = (raw / norm) if norm > 0 else 0.0

    # Reshape back to the original batch dimensions (everything except for the vocabulary axis)
    out = out.reshape(batch_shape)
    return out


def rank_turbulence_metric(alpha=1.0):
    """
    Returns a callable that computes rank-turbulence divergence with fixed alpha.
    Suitable for use with sklearn's KMedoids, which expects a metric (f1, f2).
    
    Parameters:
    -----------
    alpha : float, default=1.0
        Alpha parameter for rank-turbulence divergence
    
    Returns:
    --------
    function
        Metric function compatible with sklearn KMedoids: metric(f1, f2)
    """
    def metric(f1, f2, _alpha=alpha, **kwargs):  # capture alpha by value; **kwargs for sklearn
        return rank_turbulence(f1, f2, alpha=_alpha, axis=0)
    return metric

## Main Analysis: Multilingual Corpora (EN, DE, FR)

This section evaluates clustering performance across three languages under a characteristic probability transformation and multiple distance metrics.

In [ ]:
# Configuration
LANGUAGES = ['EN', 'DE', 'FR']
N_CLUSTERS = 25
LOG_SEQUENCE = np.logspace(1, 4, num=30, base=10, dtype=int)  # MFW from 10 to 10^4

# Distance metrics under evaluation (jensen_shannon_safe used for zero handling)
METRICS = {
    'l1': 'l1',
    'l2': 'l2', 
    'cosine': 'cosine',
    'jensen_shannon': jensen_shannon_safe,
    'rankturbulence': rank_turbulence_metric(alpha=1.0)
}

# Process each language
for lang in LANGUAGES:
    print(f"\nProcessing {lang} corpus...")
    
    # Load data
    df = pd.read_json(f'data/TfMatrix{lang}.json')
    booknames = df.columns[2:]  # Skip 'Count' and 'Word' columns
    authors = pd.read_json(f'data/authors{lang}.json', orient='index').values.reshape(1, -1)[0]
    
    # TRANSFORMATION

    # Relative frequencies
    # df[booknames] = df[booknames]

    # Centered Standard Scaler
    # z_scores = ((df[booknames].T - df[booknames].mean(axis=1)) / df[booknames].std(axis=1).T).T
    # df[booknames] = z_scores

    # Uncentered Standard Scaler
    ucd_z_scores = (df[booknames].T / df[booknames].std(axis=1).T).T
    df[booknames] = ucd_z_scores
    
    # Initialise results storage
    scores_weighted = {metric_name: [] for metric_name in METRICS.keys()}
    
    # Evaluate clustering over vocabulary size (MFW schedule)
    for mfw in tqdm.tqdm(LOG_SEQUENCE, desc=f"{lang} progress"):
        # Top MFW (most frequent words) submatrix
        data_subset = df[booknames][:mfw].T.values
        
        for metric_name, metric_func in METRICS.items():
            # K-medoids (PAM) clustering
            kmedoids = KMedoids(
                n_clusters=N_CLUSTERS, 
                random_state=RANDOM_STATE, 
                metric=metric_func, 
                method='pam'
            )
            kmedoids.fit(data_subset)
            labels = kmedoids.labels_
            
            # Clustering quality: adjusted Rand index
            ari_score = adjusted_rand_score(authors, labels)
            scores_weighted[metric_name].append(ari_score)
    
    # Write results
    results_df = pd.DataFrame(scores_weighted, index=LOG_SEQUENCE)
    results_df.index.name = 'mfw'
    output_path = f'results/uncentered_zscore_unigram_{lang}.json'
    os.makedirs('results', exist_ok=True)
    results_df.to_json(output_path)
    print(f"Results saved to {output_path}")


Processing EN corpus...


EN progress: 100%|██████████| 30/30 [01:03<00:00,  2.13s/it]


Results saved to results/uncentered_zscore_unigram_EN.json

Processing DE corpus...


DE progress: 100%|██████████| 30/30 [01:04<00:00,  2.14s/it]


Results saved to results/uncentered_zscore_unigram_DE.json

Processing FR corpus...


FR progress: 100%|██████████| 30/30 [01:02<00:00,  2.09s/it]

Results saved to results/uncentered_zscore_unigram_FR.json


## Rank-Turbulence Divergence: Alpha Parameter Sensitivity

Clustering performance is evaluated with rank-turbulence divergence (RTD) over a range of the alpha parameter. The parameter alpha governs sensitivity to rank differences: smaller alpha emphasises differences among top-ranked types; larger alpha spreads sensitivity across the full rank spectrum.

In [5]:
# Configuration
LANGUAGES = ['EN', 'DE', 'FR']
N_CLUSTERS = 25
LOG_SEQUENCE = np.logspace(1, 4, num=30, base=10, dtype=int)  # MFW from 10 to 10^4

# RTD alpha sensitivity: PAM with rank-turbulence divergence over a range of alpha values
RTD_ALPHAS = [1/12, 2/12, 3/12, 4/12, 5/12, 6/12, 8/12, 1, 2, 5]

for lang in LANGUAGES:
    print(f"\nProcessing {lang} corpus with RTD alpha variation...")
    df = pd.read_json(f'data/TfMatrix{lang}.json')
    booknames = df.columns[2:]
    authors = pd.read_json(f'data/authors{lang}.json', orient='index').values.reshape(1, -1)[0]

    # TRANSFORMATION
    # Relative frequencies
    # df[booknames] = df[booknames]

    # Centered Standard Scaler
    # z_scores = ((df[booknames].T - df[booknames].mean(axis=1)) / df[booknames].std(axis=1).T).T
    # df[booknames] = z_scores

    # Uncentered Standard Scaler
    ucd_z_scores = (df[booknames].T / df[booknames].std(axis=1).T).T
    df[booknames] = ucd_z_scores

    scores_rtd = {alpha: [] for alpha in RTD_ALPHAS}
    for mfw in tqdm.tqdm(LOG_SEQUENCE, desc=f"{lang} RTD progress"):
        data_subset = df[booknames][:mfw].T.values
        for alpha in RTD_ALPHAS:
            rtd_metric = rank_turbulence_metric(alpha=alpha)
            kmedoids = KMedoids(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, metric=rtd_metric, method='pam')
            kmedoids.fit(data_subset)
            ari_score = adjusted_rand_score(authors, kmedoids.labels_)
            scores_rtd[alpha].append(ari_score)

    results_df = pd.DataFrame(scores_rtd, index=LOG_SEQUENCE)
    results_df.index.name = 'mfw'
    output_path = f'results/uncentered_zscore_unigram_rtd_alphas_{lang}.json'
    os.makedirs('results', exist_ok=True)
    results_df.to_json(output_path)
    print(f"Results saved to {output_path}")


Processing EN corpus with RTD alpha variation...


EN RTD progress: 100%|██████████| 30/30 [09:53<00:00, 19.79s/it]


Results saved to results/uncentered_zscore_unigram_rtd_alphas_EN.json

Processing DE corpus with RTD alpha variation...


DE RTD progress: 100%|██████████| 30/30 [10:09<00:00, 20.31s/it]


Results saved to results/uncentered_zscore_unigram_rtd_alphas_DE.json

Processing FR corpus with RTD alpha variation...


FR RTD progress: 100%|██████████| 30/30 [10:02<00:00, 20.08s/it]

Results saved to results/uncentered_zscore_unigram_rtd_alphas_FR.json


## Russian Corpus: Classification (k-NN)

The similar pipeline as for EN/DE/FR is applied: `TfMatrixRU.json` and `authorsRU.json` (precomputed in `russian_tf.ipynb` notebook) are loaded. The task is **author attribution as classification**. k-nearest neighbours (k-NN) is used with the same distance metrics (L1, L2, cosine, Jensen–Shannon, rank-turbulence). For each document, the k nearest neighbours (excluding the document itself) are taken and the author is predicted by majority vote (leave-one-out). In the baseline below, relative-frequency vectors are used directly; centred or uncentred standard-score variants may be obtained by applying the corresponding transformation before distance computation.

In [7]:
from sklearn.neighbors import NearestNeighbors

N_NEIGHBORS = 4  # number of neighbours for vote (excluding self)
LOG_SEQUENCE_RU = np.logspace(1, 4, num=30, base=10, dtype=int)

# Distance metrics: same set as EN/DE/FR (callables and built-in names)
def rtd_nn_metric(x, y):
    return rank_turbulence(x, y, alpha=1.0, axis=0).item()
    
METRICS_RU = {
    'l1': 'l1', 'l2': 'l2', 'cosine': 'cosine',
    'jensen_shannon': jensen_shannon_safe,
    'rankturbulence': rtd_nn_metric,
}

In [8]:
# Load Russian data and authors
lang = 'RU'
df = pd.read_json(f'data/TfMatrix{lang}.json')
booknames = df.columns[2:]  # omit metadata columns 'Count' and 'Word'
authors = pd.read_json(f'data/authors{lang}.json', orient='index').values.reshape(-1)
authors = np.asarray(authors, dtype=object)

print(f"RU: {len(booknames)} documents, {len(np.unique(authors))} authors")

# TRANSFORMATION

# Relative frequencies
df[booknames] = df[booknames]

# Centered Standard Scaler
# z_scores = ((df[booknames].T - df[booknames].mean(axis=1)) / df[booknames].std(axis=1).T).T
# df[booknames] = z_scores

# Uncentered Standard Scaler
# ucd_z_scores = (df[booknames].T / df[booknames].std(axis=1).T).T
# df[booknames] = ucd_z_scores

# Metrics for k-NN (same set as EN/DE/FR)
METRICS_RU_ALL = {
    'l1': 'l1', 'l2': 'l2', 'cosine': 'cosine',
    'jensen_shannon': jensen_shannon_safe,
    'rankturbulence': rtd_nn_metric,
}

# k-NN evaluation over MFW schedule
scores_ru = {m: {'accuracy': [], 'balanced_accuracy': [], 'ari': []} for m in METRICS_RU_ALL}
for mfw in tqdm.tqdm(LOG_SEQUENCE_RU, desc=f"{lang} k-NN progress"):
    samples = df[booknames][:mfw].T.values  # (n_docs, n_words)
    for metric_name, metric in METRICS_RU_ALL.items():
        neigh = NearestNeighbors(n_neighbors=N_NEIGHBORS + 1, metric=metric, algorithm='brute')
        neigh.fit(samples)
        # Neighbour indices (first is self, excluded)
        neighbors_indices = neigh.kneighbors(samples, return_distance=False)[:, 1:]
        # Author prediction by majority vote among neighbours
        preds = np.array([
            Counter(authors[ind]).most_common(1)[0][0] if len(ind) > 0 else authors[i]
            for i, ind in enumerate(neighbors_indices)
        ], dtype=object)
        scores_ru[metric_name]['accuracy'].append(float(accuracy_score(authors, preds)))
        scores_ru[metric_name]['balanced_accuracy'].append(float(balanced_accuracy_score(authors, preds)))
        scores_ru[metric_name]['ari'].append(float(adjusted_rand_score(authors, preds)))

# Write results
results_ru_df = pd.DataFrame(
    {(m, k): scores_ru[m][k] for m in METRICS_RU_ALL for k in ['accuracy', 'balanced_accuracy', 'ari']},
    index=LOG_SEQUENCE_RU
)
results_ru_df.index.name = 'mfw'
os.makedirs('results', exist_ok=True)
out_path = f'results/freq_unigram_{lang}.json'
results_ru_df.to_json(out_path)
print(f"Results saved to {out_path}")

RU: 639 documents, 89 authors


RU k-NN progress: 100%|██████████| 30/30 [37:32<00:00, 75.09s/it] 

Results saved to results/freq_unigram_RU.json
